In [2]:
import glob
import re
from pathlib import Path

import polars as pl
import os

data_dir = Path("/Users/richardpears/Downloads/data 2")

parquet_blobs = sorted(glob.glob(str(data_dir / "*.parquet")))
len(parquet_blobs)


df= pl.concat([pl.read_parquet(blob) for blob in parquet_blobs])
df.shape

(770623, 13)

In [3]:
from pathlib import Path
import glob




glob_pattern= str(data_dir / "*.parquet")
# 1. Scan all files lazily and include the file path metadata
df = (
    pl.scan_parquet(glob_pattern, include_file_paths="file_path")
    
    # 2. Extract just the symbol from the path string using a Polars expression
    # This strips out everything before the filename and drops the ".parquet" extension
    .with_columns(
        pl.col("file_path")
        .str.split(os.sep)       # Split path by system folder separator (/ or \)
        .list.last()             # Get the last item (e.g., "AAPL.parquet")
        .str.replace(".parquet", "") # Strip the extension to leave just "AAPL"
        .alias("symbol")         # Name the new column 'symbol'
    )
    
    # 3. Drop the temporary full path column so your schema stays perfectly clean
    .drop("file_path")
    
    # 4. Trigger the multi-threaded parallel execution engine
    .collect()
)

print(df.shape)


(770623, 14)


In [28]:
df['symbol'].value_counts().sort("symbol")

symbol,count
str,u32
"""A""",824
"""AAPL""",8677
"""ABBV""",1976
"""ABNB""",770
"""ABT""",2155
…,…
"""XYZ""",314
"""YUM""",1563
"""ZBH""",1100


In [24]:
df

title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,symbol
str,str,datetime[μs],list[str],str,str,str,str,str,list[struct[2]],f64,str,list[struct[4]],str
"""Blood-based gene expression si…","""https://www.nature.com/article…",2016-01-05 05:51:32,"[""Hiroaki Hori"", ""Daimei Sasayama"", … ""Hiroshi Kunugi""]","""This study investigates blood-…",null,"""Nature""","""General""","""Nature""","[{""life_sciences"",""0.918616""}]",0.122718,"""Neutral""","[{""A"",""0.900629"",""0.101134"",""Neutral""}]","""A"""
"""Agilent accuses Twist Bioscien…","""https://www.biopharmadive.com/…",2016-02-05 05:51:32,"[""Nicole Gray""]","""Agilent Technologies has filed…",null,"""BioPharma Dive""","""General""","""BioPharma Dive""","[{""life_sciences"",""0.905912""}, {""technology"",""0.833628""}]",-0.63852,"""Bearish""","[{""A"",""1.000000"",""-0.639674"",""Bearish""}]","""A"""
"""Avago adopts new name""","""https://www.coloradoan.com/sto…",2016-02-16 05:51:32,"[""Adrian D. Garcia""]","""Avago Technologies, a major em…",null,"""The Coloradoan""","""General""","""The Coloradoan""","[{""mergers_and_acquisitions"",""1.000000""}, {""technology"",""0.813530""}, … {""manufacturing"",""0.633771""}]",0.176532,"""Somewhat-Bullish""","[{""A"",""0.603131"",""-0.119860"",""Neutral""}, {""AVGO"",""1.000000"",""0.440914"",""Bullish""}]","""A"""
"""Nanotechnology Company Exicure…","""https://www.biospace.com/nanot…",2016-02-26 05:51:32,"[""Mark Terry""]","""Exicure, a nanotechnology comp…",null,"""BioSpace""","""General""","""BioSpace""","[{""finance"",""0.834063""}, {""life_sciences"",""0.934113""}, {""technology"",""0.831460""}]",0.299569,"""Somewhat-Bullish""","[{""A"",""0.707012"",""0.339568"",""Somewhat-Bullish""}, {""ABBV"",""0.906186"",""0.443224"",""Bullish""}, … {""AON"",""0.646505"",""0.242678"",""Somewhat-Bullish""}]","""A"""
"""Agilent Technologies introduce…","""https://www.newfoodmagazine.co…",2016-05-11 05:51:32,"[""Agilent Technologies Inc.""]","""Agilent Technologies has intro…",null,"""New Food magazine""","""General""","""New Food magazine""","[{""life_sciences"",""0.812509""}, {""technology"",""0.729947""}, {""finance"",""0.636568""}]",0.426309,"""Bullish""","[{""A"",""1.000000"",""0.410251"",""Bullish""}]","""A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Boston Scientific (BSX) Stock …","""https://ts2.tech/en/boston-sci…",2025-12-25 03:08:20,"[""Khadija Saeed""]","""Boston Scientific (NYSE: BSX) …","""NULL""","""ts2.tech""","""General""","""ts2.tech""","[{""financial_markets"",""0.919847""}, {""earnings"",""0.831648""}, {""life_sciences"",""0.702408""}]",0.184648,"""Somewhat-Bullish""","[{""BSX"",""1.000000"",""0.300000"",""Somewhat-Bullish""}, {""ZTS"",""0.561013"",""0.049371"",""Neutral""}, … {""ABT"",""0.574339"",""0.240802"",""Somewhat-Bullish""}]","""ZTS"""
"""Yousif Capital Management LLC …","""https://www.marketbeat.com/ins…",2025-12-27 14:09:06,"[""MarketBeat""]","""Yousif Capital Management LLC …","""https://www.marketbeat.com/log…","""MarketBeat""","""General""","""MarketBeat""","[{""earnings"",""0.945007""}, {""financial_markets"",""0.802718""}, {""life_sciences"",""0.700296""}]",0.299451,"""Somewhat-Bullish""","[{""ELAN"",""1.000000"",""0.476012"",""Bullish""}, {""ZTS"",""0.648012"",""0.128038"",""Neutral""}]","""ZTS"""
"""Zoetis (ZTS) Positioned Within…","""https://finance.yahoo.com/news…",2025-12-28 02:17:00,"[""Vardah Gill""]","""Zoetis Inc. (NYSE: ZTS) is inc…","""NULL""","""Yahoo Finance""","""General""","""Yahoo Finance""","[{""life_sciences"",""0.945413""}, {""financial_markets"",""0.721864""}, {""finance"",""0.630738""}]",0.104321,"""Neutral""","[{""MS"",""0.739089"",""-0.132256"",""Neutral""}, {""ZTS"",""1.000000"",""0.293772"",""Somewhat-Bullish""}]","""ZTS"""


In [30]:
# write to a single parquet file inside a folder called compacted in the same directory as the source files


df.write_parquet(
    "/Users/richardpears/Downloads/data 2/compacted/compacted_news.parquet",
    mkdir=True  
)

In [37]:
pl.DataFrame.write_parquet?

Signature:
pl.DataFrame.write_parquet(
    self,
    file: 'str | Path | IO[bytes]',
    *,
    compression: 'ParquetCompression' = 'zstd',
    compression_level: 'int | None' = None,
    statistics: 'bool | str | dict[str, bool]' = True,
    row_group_size: 'int | None' = None,
    data_page_size: 'int | None' = None,
    use_pyarrow: 'bool' = False,
    pyarrow_options: 'dict[str, Any] | None' = None,
    partition_by: 'str | Sequence[str] | None' = None,
    partition_chunk_size_bytes: 'int' = 4294967296,
    storage_options: 'StorageOptionsDict | None' = None,
    credential_provider: "CredentialProviderFunction | Literal['auto'] | None" = 'auto',
    retries: 'int | None' = None,
    metadata: 'ParquetMetadata | None' = None,
    arrow_schema: 'ArrowSchemaExportable | None' = None,
    mkdir: 'bool' = False,
) -> 'None'
Docstring:
Write to Apache Parquet file.

Parameters
----------
file
    File path or writable file-like object to which the result will be written.
    This shoul

In [15]:
df = df.with_columns(pl.col("time_published").dt.strftime("%Y-%m").alias("year_month_published"))



In [24]:
df['year_month_published'].value_counts().sort("year_month_published").tail(15)

year_month_published,count
str,u32
"""2024-10""",135
"""2024-11""",143
"""2024-12""",159
"""2025-01""",86
"""2025-02""",129
…,…
"""2025-08""",208
"""2025-09""",250
"""2025-10""",3962


In [19]:
df

title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,year_month_published
str,str,datetime[μs],list[str],str,str,str,str,str,list[struct[2]],f64,str,list[struct[4]],str
"""Six Flags Plans to Open Theme …","""https://fortune.com/2016/06/20…",2016-01-01 00:00:00,"[""Reuters""]","""Six Flags Entertainment Corp. …",null,"""Fortune""","""General""","""Fortune""","[{""retail_wholesale"",""0.906395""}, {""economy_fiscal"",""0.722514""}]",0.139805,"""Neutral""","[{""FUN"",""0.334564"",""0.032962"",""Neutral""}, {""MSFT"",""0.643501"",""0.219803"",""Somewhat-Bullish""}]","""2016-01"""
"""Mohammed receives Microsoft CE…","""https://www.emirates247.com/ne…",2016-01-05 07:01:44,"[""WAM""]","""His Highness Sheikh Mohammed b…",null,"""Emirates24|7""","""General""","""Emirates24|7""","[{""technology"",""0.844086""}]",0.425728,"""Bullish""","[{""MSFT"",""1.000000"",""0.405680"",""Bullish""}]","""2016-01"""
"""Microsoft Surface Pro 4 finall…","""https://indianexpress.com/arti…",2016-01-08 10:44:00,"[""Debashis Sarkar""]","""Microsoft has officially launc…",null,"""The Indian Express""","""General""","""The Indian Express""","[{""technology"",""0.918957""}]",0.219815,"""Somewhat-Bullish""","[{""MSFT"",""0.954498"",""0.348739"",""Somewhat-Bullish""}, {""AMZN"",""0.651623"",""0.138060"",""Neutral""}]","""2016-01"""
"""Gates and Bezos partner with S…","""https://www.bizjournals.com/ne…",2016-01-11 06:37:27,"[""Anthony Noto""]","""Bill Gates and Jeff Bezos are …",null,"""The Business Journals""","""General""","""The Business Journals""","[{""life_sciences"",""0.923686""}, {""technology"",""0.808673""}]",0.343826,"""Somewhat-Bullish""","[{""GRAL"",""1.000000"",""0.449829"",""Bullish""}, {""MSFT"",""0.611866"",""0.292967"",""Somewhat-Bullish""}, {""AMZN"",""0.629024"",""0.257760"",""Somewhat-Bullish""}]","""2016-01"""
"""Jeff Bezos and Bill Gates bet …","""https://www.marketwatch.com/st…",2016-01-12 13:58:00,[],"""Jeff Bezos and Bill Gates have…",null,"""MarketWatch""","""General""","""MarketWatch""","[{""life_sciences"",""0.937820""}, {""technology"",""0.815812""}]",0.310524,"""Somewhat-Bullish""","[{""GRAL"",""1.000000"",""0.409359"",""Bullish""}, {""AMZN"",""0.712384"",""0.274203"",""Somewhat-Bullish""}, {""MSFT"",""0.667885"",""0.272035"",""Somewhat-Bullish""}]","""2016-01"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Hewlett Packard Enterprise sto…","""https://www.ad-hoc-news.de/boe…",2025-12-31 19:08:00,"[""NULL""]","""Hewlett Packard Enterprise sto…","""NULL""","""AD HOC NEWS""","""General""","""AD HOC NEWS""","[{""financial_markets"",""0.946677""}, {""technology"",""0.801919""}, {""earnings"",""0.719557""}]",0.167414,"""Somewhat-Bullish""","[{""HPE"",""0.976074"",""0.263618"",""Somewhat-Bullish""}, {""IBM"",""0.645176"",""0.107005"",""Neutral""}, … {""AMZN"",""0.622983"",""0.137511"",""Neutral""}]","""2025-12"""
"""The Best Tech Stocks to Buy in…","""https://finviz.com/news/265420…",2025-12-31 20:07:37,"[""Justin Pope""]","""This article identifies three …","""https://g.foolcdn.com/image/?u…","""Finviz""","""General""","""Finviz""","[{""technology"",""0.934423""}, {""financial_markets"",""0.832460""}, {""earnings"",""0.714399""}]",0.435063,"""Bullish""","[{""ADP"",""1.000000"",""0.393972"",""Bullish""}, {""MSFT"",""0.994874"",""0.442568"",""Bullish""}, {""MSI"",""0.912444"",""0.378551"",""Bullish""}]","""2025-12"""
"""Prediction: This Company Could…","""https://www.fool.com/investing…",2025-12-31 20:08:11,"[""Keithen Drury""]","""Nvidia is currently the larges…","""https://g.foolcdn.com/image/?u…","""The Motley Fool""","""General""","""The Motley Fool""","[{""technology"",""0.943093""}, {""financial_markets"",""0.845075""}]",0.115555,"""Neutral""","[{""NVDA"",""1.000000"",""0.121601"",""Neutral""}, {""GOOGL"",""0.924239"",""0.300397"",""Somewhat-Bullish""}, … {""MSFT"",""0.614398"",""0.115408"",""Neutral""}]","""2025-12"""
